# Parcelwise Welch t-tests

This notebook uses every subject with complete graph outputs in `outputs/`. It reports the cohort actually available, so these results are not a substitute for the planned, cohort-validated 267-subject run.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'data' / 'database_finale_labels_corrette.csv').is_file()
)
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
DATABASE = PROJECT_ROOT / 'data' / 'database_finale_labels_corrette.csv'
STATISTICS_DIR = OUTPUT_ROOT / 'statistics'
FEATURE_LOCATIONS = {
    'weighted_degree_expW': ('wasserstein_graphs_expW', 'weighted_degree.dat'),
    'weighted_degree_inv1pW': ('wasserstein_graphs_inv1pW', 'weighted_degree.dat'),
}

feature_subjects = {
    feature: {
        subject_dir.name
        for subject_dir in (OUTPUT_ROOT / directory).iterdir()
        if subject_dir.is_dir()
        and (subject_dir / filename).is_file()
        and (subject_dir / 'parcel_order.txt').is_file()
    }
    for feature, (directory, filename) in FEATURE_LOCATIONS.items()
}
subjects = sorted(set.intersection(*feature_subjects.values()))
if not subjects:
    raise FileNotFoundError('No subjects have complete graph outputs for both endpoints.')

database = pd.read_csv(DATABASE).copy()
database['subject_id'] = 'sub-' + database['OASISID'].astype(str).str.removeprefix('OAS3')
cohort = database.set_index('subject_id').reindex(subjects)
if cohort['HStatus'].isna().any():
    missing = cohort.index[cohort['HStatus'].isna()].tolist()
    raise ValueError(f'Missing cohort labels for: {missing[:10]}')
if cohort['HStatus'].nunique() != 2:
    raise ValueError('The available outputs do not contain both cohort groups.')

pd.DataFrame({
    'subjects_with_complete_outputs': [len(subjects)],
    **{f'n_{group.lower()}': [count] for group, count in cohort['HStatus'].value_counts().items()},
})

,subjects_with_complete_outputs,n_healthy,n_unhealthy
0,153,137,16


In [ ]:
def load_feature_matrix(feature):
    directory, filename = FEATURE_LOCATIONS[feature]
    root = OUTPUT_ROOT / directory
    # Subjects were produced by two pipeline versions that emit label blocks in
    # different orders (numeric vs lexicographic), so each subject's
    # weighted_degree.dat is aligned to ITS OWN parcel_order.txt. Reindex every
    # subject onto one canonical order before stacking; identical byte size would
    # otherwise hide the misalignment and silently corrupt the t-tests.
    canonical = sorted((root / subjects[0] / 'parcel_order.txt').read_text().splitlines())
    canon_pos = {pid: i for i, pid in enumerate(canonical)}
    n_parcels = len(canonical)
    expected_bytes = n_parcels * np.dtype(np.float64).itemsize
    matrix = np.empty((len(subjects), n_parcels), dtype=np.float64)
    for row, subject in enumerate(subjects):
        order = (root / subject / 'parcel_order.txt').read_text().splitlines()
        if len(order) != n_parcels:
            raise ValueError(f'Parcel count mismatch for {subject}: {len(order)} != {n_parcels}')
        path = root / subject / filename
        if path.stat().st_size != expected_bytes:
            raise ValueError(f'Feature size mismatch: {path}')
        vector = np.fromfile(path, dtype=np.float64)
        try:
            perm = np.fromiter((canon_pos[pid] for pid in order), dtype=np.int64, count=n_parcels)
        except KeyError as exc:
            raise ValueError(f'Subject {subject} has a parcel absent from the canonical set: {exc}') from exc
        matrix[row, perm] = vector
    if not np.isfinite(matrix).all():
        raise ValueError(f'Non-finite values found in {feature}')
    return canonical, matrix

def welch_results(feature):
    parcel_ids, matrix = load_feature_matrix(feature)
    healthy = matrix[cohort['HStatus'].eq('Healthy').to_numpy()]
    unhealthy = matrix[cohort['HStatus'].eq('Unhealthy').to_numpy()]
    t_stat, p_value = stats.ttest_ind(healthy, unhealthy, axis=0, equal_var=False)
    p_fdr_bh = multipletests(p_value, alpha=0.05, method='fdr_bh')[1]
    return pd.DataFrame({
        'parcel_id': parcel_ids,
        'healthy_mean': healthy.mean(axis=0),
        'unhealthy_mean': unhealthy.mean(axis=0),
        'unhealthy_minus_healthy': unhealthy.mean(axis=0) - healthy.mean(axis=0),
        'welch_t_healthy_minus_unhealthy': t_stat,
        'p_value': p_value,
        'p_fdr_bh': p_fdr_bh,
        'significant_fdr_05': p_fdr_bh < 0.05,
    })

results = {feature: welch_results(feature) for feature in FEATURE_LOCATIONS}
summary = pd.DataFrame([
    {
        'feature': feature,
        'n_subjects': len(subjects),
        'n_parcels': len(frame),
        'welch_bh_fdr_05': int(frame['significant_fdr_05'].sum()),
    }
    for feature, frame in results.items()
]).set_index('feature')
summary

In [3]:
# Parcel-ID overlap across the two graph representations.
significant_sets = {
    feature: set(frame.loc[frame['significant_fdr_05'], 'parcel_id'])
    for feature, frame in results.items()
}
overlap = pd.DataFrame({
    right: {left: len(significant_sets[left] & significant_sets[right]) for left in significant_sets}
    for right in significant_sets
})
overlap

,weighted_degree_expW,weighted_degree_inv1pW
weighted_degree_expW,3,2
weighted_degree_inv1pW,2,2


In [4]:
# Inspect the strongest effects in the inverse-1-plus-W representation.
results['weighted_degree_inv1pW'].sort_values('p_fdr_bh').loc[:, [
    'parcel_id', 'healthy_mean', 'unhealthy_mean', 'unhealthy_minus_healthy',
    'welch_t_healthy_minus_unhealthy', 'p_value', 'p_fdr_bh',
]].head(25)

,parcel_id,healthy_mean,unhealthy_mean,unhealthy_minus_healthy,welch_t_healthy_minus_unhealthy,p_value,p_fdr_bh
22852,label_24/roi_51631,0.702236,0.761836,0.059601,-6.570144,1.057787e-08,0.000639
42291,label_3/roi_5414,0.713736,0.758114,0.044379,-5.837554,3.208272e-08,0.000969
29872,label_3/roi_14805,0.700106,0.747541,0.047435,-5.308325,3.152966e-06,0.056741
30561,label_3/roi_15431,0.722701,0.758703,0.036002,-4.990181,3.757995e-06,0.056741
28404,label_3/roi_13470,0.708073,0.755008,0.046935,-4.939460,1.073694e-05,0.129692
21596,label_24/roi_50375,0.728985,0.759170,0.030185,-4.673458,1.364103e-05,0.137308
19657,label_24/roi_48436,0.548653,0.491852,-0.056801,4.526219,3.741627e-05,0.171238
46340,label_3/roi_9463,0.739378,0.765821,0.026443,-4.314572,4.489448e-05,0.171238
24390,label_3/roi_0804,0.730888,0.760967,0.030079,-4.353844,5.356212e-05,0.171238
22933,label_24/roi_51712,0.737721,0.765261,0.027540,-4.513801,2.855352e-05,0.171238


In [5]:
STATISTICS_DIR.mkdir(parents=True, exist_ok=True)
for feature, frame in results.items():
    frame.to_csv(STATISTICS_DIR / f'{feature}_parcel_statistics.csv', index=False)
summary.to_csv(STATISTICS_DIR / 'feature_summary.csv')
overlap.to_csv(STATISTICS_DIR / 'significant_parcel_overlap.csv')
STATISTICS_DIR

PosixPath('/home/lucagalli/Projects/parcellating_dbm/outputs/statistics')

## Age/sex-matched sanity check

The cohort above is heavily unbalanced (137 healthy vs 16 unhealthy) because only 170 of the
1188 available subjects have made it through the Jacobian/graph stages so far, and that subset
happens to be mostly healthy. Group-size imbalance itself doesn't invalidate Welch's t-test, but
it does mean the healthy mean is estimated far more precisely than the unhealthy mean, and any
age/sex confound in the imbalanced pool isn't controlled for.

As a quick sanity check (not a substitute for processing the full cohort), match each of the 16
unhealthy subjects to one healthy subject from the 137 available, using the same greedy
same-sex-then-closest-age strategy as `_balance_unhealthy` in `scripts/data_handle.py`. This gives
a balanced 16 vs 16 comparison drawn from data we already have.

In [6]:
import csv

age_sex = {}
with open(DATABASE, newline='') as f:
    for row in csv.DictReader(f):
        sid = 'sub-' + row['OASISID'].removeprefix('OAS3')
        if sid in set(subjects) and row['age at visit'].strip():
            age_sex[sid] = {'age': float(row['age at visit']), 'sex': row['GENDER']}

missing_age_sex = [s for s in subjects if s not in age_sex]
if missing_age_sex:
    raise ValueError(f'Missing age/sex for: {missing_age_sex}')

unhealthy_ids = [s for s in subjects if cohort.loc[s, 'HStatus'] == 'Unhealthy']
healthy_ids = [s for s in subjects if cohort.loc[s, 'HStatus'] == 'Healthy']

available = set(healthy_ids)
matched_healthy = []
for uid in unhealthy_ids:
    u = age_sex[uid]
    same_sex = [hid for hid in available if age_sex[hid]['sex'] == u['sex']]
    candidates = same_sex if same_sex else list(available)
    best = min(candidates, key=lambda hid: abs(age_sex[hid]['age'] - u['age']))
    matched_healthy.append(best)
    available.remove(best)

matched_subjects = matched_healthy + unhealthy_ids
print(f'Matched cohort: {len(matched_healthy)} healthy vs {len(unhealthy_ids)} unhealthy')

pd.DataFrame({
    'unhealthy': unhealthy_ids,
    'unhealthy_age': [age_sex[u]['age'] for u in unhealthy_ids],
    'unhealthy_sex': [age_sex[u]['sex'] for u in unhealthy_ids],
    'matched_healthy': matched_healthy,
    'matched_age': [age_sex[h]['age'] for h in matched_healthy],
    'matched_sex': [age_sex[h]['sex'] for h in matched_healthy],
    'age_diff': [abs(age_sex[h]['age'] - age_sex[u]['age']) for h, u in zip(matched_healthy, unhealthy_ids)],
})

Matched cohort: 16 healthy vs 16 unhealthy


,unhealthy,unhealthy_age,unhealthy_sex,matched_healthy,matched_age,matched_sex,age_diff
0,sub-0025,64.86,Female,sub-0251,64.78,Female,0.08
1,sub-0027,69.21,Male,sub-1042,69.20,Male,0.01
2,sub-0028,68.05,Male,sub-0625,68.11,Male,0.06
3,sub-0029,72.42,Male,sub-0542,72.43,Male,0.01
4,sub-0052,59.06,Female,sub-0792,58.99,Female,0.07
5,sub-0063,70.30,Female,sub-1380,70.27,Female,0.03
6,sub-0104,67.81,Female,sub-1203,67.94,Female,0.13
7,sub-0108,64.85,Female,sub-1211,64.94,Female,0.09
8,sub-0111,59.72,Female,sub-0093,59.61,Female,0.11
9,sub-0114,68.77,Female,sub-1307,68.68,Female,0.09


In [7]:
def welch_results_subset(feature, subject_subset):
    parcel_ids, matrix = load_feature_matrix(feature)
    row_index = {subject: i for i, subject in enumerate(subjects)}
    rows = [row_index[s] for s in subject_subset]
    sub_matrix = matrix[rows]
    sub_cohort = cohort.reindex(subject_subset)
    healthy = sub_matrix[sub_cohort['HStatus'].eq('Healthy').to_numpy()]
    unhealthy = sub_matrix[sub_cohort['HStatus'].eq('Unhealthy').to_numpy()]
    t_stat, p_value = stats.ttest_ind(healthy, unhealthy, axis=0, equal_var=False)
    p_fdr_bh = multipletests(p_value, alpha=0.05, method='fdr_bh')[1]
    return pd.DataFrame({
        'parcel_id': parcel_ids,
        'healthy_mean': healthy.mean(axis=0),
        'unhealthy_mean': unhealthy.mean(axis=0),
        'unhealthy_minus_healthy': unhealthy.mean(axis=0) - healthy.mean(axis=0),
        'welch_t_healthy_minus_unhealthy': t_stat,
        'p_value': p_value,
        'p_fdr_bh': p_fdr_bh,
        'significant_fdr_05': p_fdr_bh < 0.05,
    })

matched_results = {feature: welch_results_subset(feature, matched_subjects) for feature in FEATURE_LOCATIONS}
matched_summary = pd.DataFrame([
    {
        'feature': feature,
        'n_subjects': len(matched_subjects),
        'n_parcels': len(frame),
        'welch_bh_fdr_05': int(frame['significant_fdr_05'].sum()),
    }
    for feature, frame in matched_results.items()
]).set_index('feature')

comparison = summary.join(matched_summary, lsuffix='_full_unbalanced', rsuffix='_matched_16v16')
comparison

,n_subjects_full_unbalanced,n_parcels_full_unbalanced,welch_bh_fdr_05_full_unbalanced,n_subjects_matched_16v16,n_parcels_matched_16v16,welch_bh_fdr_05_matched_16v16
feature,,,,,,
weighted_degree_expW,153,60395,3,32,60395,0
weighted_degree_inv1pW,153,60395,2,32,60395,0
